# POC: GPT-4o Vision Direct  vs  Azure DI + LLM
**Goal:** Extract structured JSON from contractor daily report PDFs using both approaches and compare.

```
Approach A: PDF → images → GPT-4o Vision → JSON
Approach B: PDF → Azure DI (raw text) → GPT-4o → JSON
```

## 0. Install Dependencies

In [1]:
# Run once
# pdf2image needs poppler installed: `brew install poppler` (mac) or `apt install poppler-utils` (linux)
%pip install openai azure-ai-documentintelligence pdf2image pillow python-dotenv

Note: you may need to restart the kernel to use updated packages.


d:\Ddrive\wt\codee\poc\extraction_poc2\poc_ai_extraction\.venv\Scripts\python.exe: No module named pip


## 1. Config — Fill Your Keys Here

In [ ]:
import os

# ---- Azure OpenAI ----
AZURE_OPENAI_ENDPOINT = "https://<your-resource>.openai.azure.com/"  # base domain only, no path
AZURE_OPENAI_KEY      = "<your-key>"
AZURE_OPENAI_DEPLOYMENT = "gpt-4o"          # your deployment name
AZURE_OPENAI_API_VERSION = "2024-02-01"

# ---- Azure Document Intelligence ----
DI_ENDPOINT = "https://<your-di-resource>.cognitiveservices.azure.com/"
DI_KEY      = "<your-di-key>"

# ---- Test PDF ----
PDF_PATH = "24_12_16_HJK_Dalies.pdf"        # drop your PDF here

## 2. Target JSON Schema
Same schema for both approaches — so comparison is apples-to-apples.

In [ ]:
# The schema we want out of BOTH approaches
TARGET_SCHEMA = """
{
  "Vendor": "string",
  "Date": "YYYY-MM-DD",
  "TotalWorkers": number,
  "TotalWorkHr": number
}
"""

EXTRACTION_PROMPT = f"""
You are extracting data from a contractor daily work report.
Return ONLY valid JSON matching this schema (one object per report page):
{TARGET_SCHEMA}

Rules:
- Vendor = contractor company name
- TotalWorkers = count of individual workers listed
- TotalWorkHr  = total manhours (use the printed total if available, else sum individual hours)
- If a field is missing, use null
- Return a JSON array if multiple pages/reports are present
"""

---
## APPROACH A — GPT-4o Vision Direct
```
PDF → convert each page to image → send images to GPT-4o → JSON
```

In [ ]:
import base64, json
from pdf2image import convert_from_path
from openai import AzureOpenAI

# Step 1: PDF → images
pages = convert_from_path(PDF_PATH, dpi=200)   # 200 dpi is sufficient for forms
print(f"PDF has {len(pages)} page(s)")

# Step 2: Convert images to base64
def image_to_base64(pil_image):
    from io import BytesIO
    buf = BytesIO()
    pil_image.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

images_b64 = [image_to_base64(p) for p in pages]
print(f"Converted {len(images_b64)} page(s) to base64")

In [ ]:
# Step 3: Send to GPT-4o Vision
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

# Build content: one image block per page
content = [{"type": "text", "text": EXTRACTION_PROMPT}]
for b64 in images_b64:
    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "high"}
    })

response_a = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[{"role": "user", "content": content}],
    max_tokens=1000
)

raw_a = response_a.choices[0].message.content
print("=== APPROACH A — GPT-4o Vision Raw Response ===")
print(raw_a)

In [ ]:
# Parse JSON from response
import re

def parse_json(text):
    # Strip markdown code fences if present
    text = re.sub(r"```json|```", "", text).strip()
    return json.loads(text)

result_a = parse_json(raw_a)
print("=== APPROACH A — Parsed JSON ===")
print(json.dumps(result_a, indent=2))

---
## APPROACH B — Azure DI (prebuilt-layout) → GPT-4o
```
PDF → Azure DI extracts raw text → GPT-4o structures it → JSON
```

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

# Step 1: Send PDF to Azure DI
di_client = DocumentIntelligenceClient(
    endpoint=DI_ENDPOINT,
    credential=AzureKeyCredential(DI_KEY)
)

with open(PDF_PATH, "rb") as f:
    poller = di_client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=f,
        content_type="application/pdf"
    )

di_result = poller.result()

# Step 2: Extract raw text from DI result
raw_text = di_result.content   # all text DI found, in reading order
print("=== Azure DI Raw Text ===")
print(raw_text[:1500], "...")  # preview first 1500 chars

In [ ]:
# Step 3: Pass DI raw text to GPT-4o for structuring
response_b = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {"role": "user", "content": f"{EXTRACTION_PROMPT}\n\nDocument text:\n{raw_text}"}
    ],
    max_tokens=1000
)

raw_b = response_b.choices[0].message.content
print("=== APPROACH B — GPT-4o Raw Response ===")
print(raw_b)

In [ ]:
result_b = parse_json(raw_b)
print("=== APPROACH B — Parsed JSON ===")
print(json.dumps(result_b, indent=2))

---
## Side-by-Side Comparison

In [ ]:
print("APPROACH A — GPT-4o Vision Direct")
print(json.dumps(result_a, indent=2))

print("\nAPPROACH B — Azure DI + GPT-4o")
print(json.dumps(result_b, indent=2))

print("\n--- Token Usage ---")
print(f"A (Vision): input={response_a.usage.prompt_tokens}, output={response_a.usage.completion_tokens}")
print(f"B (DI+LLM): input={response_b.usage.prompt_tokens}, output={response_b.usage.completion_tokens}")
print("Note: Approach B also incurred Azure DI cost on top of LLM tokens")